## Large-Scale Matching with Persistent Index files

Uses faiss's ability to save and load indices from disk to avoid recomputing matches from scratch each time new treated units arrive. This considerably speeds up the matching process in production settings since at the point of computing effects, only lookup costs are incurred instead of full index construction and search.

In [1]:
import pandas as pd
import numpy as np
from fastmatch import Matching
from sklearn.linear_model import LinearRegression

In [2]:
lalonde_obs = pd.read_parquet("lalonde_obs.parquet")
w, y = lalonde_obs.treatment.values, lalonde_obs.earnings1978.values
X = lalonde_obs.drop(columns=["treatment", "earnings1978"]).values

print(f"\nDataset: {lalonde_obs.shape[0]} observations, {X.shape[1]} covariates")
print(f"  Treatment: {w.sum()} treated, {(1-w).sum()} control")


Dataset: 16177 observations, 8 covariates
  Treatment: 185.0 treated, 15992.0 control


## FAISS

### Initial Analysis with Index Saving

The first call to `fit` creates the index that we can then save.


In [3]:
%%time
m = Matching("ATT", k=5, backend="faiss", bias_corr_mod=LinearRegression())
est, se = m.fit(y, w, X)
print(f"\nATT Estimate: ${est:.2f} (SE: ${se:.2f})")


ATT Estimate: $1510.86 (SE: $693.15)
CPU times: user 2.06 s, sys: 3.59 ms, total: 2.07 s
Wall time: 143 ms


In [4]:
!rm saved_indices/*

In [5]:
m.save_indices("./saved_indices/lalonde_control_pop")
print("\nSaved indices to: ./saved_indices/")
print("  This creates timestamped index and metadata files for reuse")


Saved indices to: ./saved_indices/
  This creates timestamped index and metadata files for reuse


This creates objects for the control index, metadata, and pickles the bias-correction scikit model for future use.

In [6]:
!ls saved_indices/

lalonde_control_pop_biasmod_20251025_192228.pkl
lalonde_control_pop_control_faiss_20251025_192228.index
lalonde_control_pop_control_faiss_20251025_192228.meta


### Fast Reanalysis with Pre-loaded Indices

In [7]:
import os

control_files = [
    f for f in os.listdir("./saved_indices") if "control" in f and f.endswith(".index")
]
control_path = os.path.join("./saved_indices", control_files[0][:-6])
bias_files = [f for f in os.listdir("./saved_indices") if f.endswith(".pkl")]
bias_path = os.path.join("./saved_indices", bias_files[0]) if bias_files else None

print(f"Loading pre-built index from: {control_path}")
print(f"Loading model from: {bias_path}")

Loading pre-built index from: ./saved_indices/lalonde_control_pop_control_faiss_20251025_192228
Loading model from: ./saved_indices/lalonde_control_pop_biasmod_20251025_192228.pkl


In [8]:
%%time
m_loaded = Matching("ATT", k=5, backend="faiss")
m_loaded.load_indices(
    control_index_path=control_path,
    bias_model_path=bias_path,
)

print("Index loaded! No re-indexing needed.")

Index loaded! No re-indexing needed.
CPU times: user 776 μs, sys: 991 μs, total: 1.77 ms
Wall time: 1.4 ms


In [9]:
%%time
est_loaded, se_loaded = m_loaded.estimate_with_preloaded(y, w, X)

CPU times: user 1.89 s, sys: 6.36 ms, total: 1.89 s
Wall time: 166 ms


In [10]:
print(f"\nATT Estimate (from loaded index): ${est_loaded:.2f} (SE: ${se_loaded:.2f})")

print("\nVerification:")
print(f"  Original: ${est:.2f}")
print(f"  Reloaded: ${est_loaded:.2f}")
print(f"  Match: {np.isclose(est, est_loaded)}")


ATT Estimate (from loaded index): $1510.86 (SE: $693.15)

Verification:
  Original: $1510.86
  Reloaded: $1510.86
  Match: True


### updating index with new observations

In [11]:
n_new_treated = 50
new_treated_indices = np.random.choice(
    np.where(w == 1)[0], n_new_treated, replace=False
)

y_new_treated = y[new_treated_indices]
X_new_treated = X[new_treated_indices]

control_indices = np.where(w == 0)[0]
full_data_w = np.concatenate([np.ones(n_new_treated), np.zeros(len(control_indices))])
full_data_y = np.concatenate([y_new_treated, y[control_indices]])
full_data_X = np.vstack([X_new_treated, X[control_indices]])

est_new, se_new = m_loaded.estimate_with_preloaded(
    full_data_y, full_data_w, full_data_X
)
print(f"\nATT for new cohort: ${est_new:.2f} (SE: ${se_new:.2f})")
print("No re-indexing of control population needed!")


ATT for new cohort: $1519.87 (SE: $1567.57)
No re-indexing of control population needed!


In [12]:
print("=" * 70)
print("Index files created:")
print("=" * 70)
for f in os.listdir("./saved_indices"):
    size = os.path.getsize(os.path.join("./saved_indices", f)) / 1024
    print(f"  {f}: {size:.1f} KB")


Index files created:
  lalonde_control_pop_control_faiss_20251025_192228.meta: 1.8 KB
  lalonde_control_pop_control_faiss_20251025_192228.index: 499.8 KB
  lalonde_control_pop_biasmod_20251025_192228.pkl: 0.5 KB


## Voyager

### Initial Analysis with Index Saving

The first call to `fit` creates the index that we can then save.


In [13]:
%%time
m = Matching("ATT", k=5, backend="voyager", bias_corr_mod=LinearRegression())
est, se = m.fit(y, w, X)
print(f"\nATT Estimate: ${est:.2f} (SE: ${se:.2f})")


ATT Estimate: $1696.51 (SE: $708.63)
CPU times: user 1.56 s, sys: 45.2 ms, total: 1.61 s
Wall time: 127 ms


In [14]:
!rm saved_indices/*

In [15]:
m.save_indices("./saved_indices/lalonde_control_pop")
print("\nSaved indices to: ./saved_indices/")
print("  This creates timestamped index and metadata files for reuse")


Saved indices to: ./saved_indices/
  This creates timestamped index and metadata files for reuse


This creates objects for the control index, metadata, and pickles the bias-correction scikit model for future use.

In [16]:
!ls saved_indices/

lalonde_control_pop_biasmod_20251025_192229.pkl
lalonde_control_pop_control_voyager_20251025_192229.index
lalonde_control_pop_control_voyager_20251025_192229.meta


### Fast Reanalysis with Pre-loaded Indices

In [17]:
import os

control_files = [
    f for f in os.listdir("./saved_indices") if "control" in f and f.endswith(".index")
]
control_path = os.path.join("./saved_indices", control_files[0][:-6])
bias_files = [f for f in os.listdir("./saved_indices") if f.endswith(".pkl")]
bias_path = os.path.join("./saved_indices", bias_files[0]) if bias_files else None

print(f"Loading pre-built index from: {control_path}")
print(f"Loading model from: {bias_path}")

Loading pre-built index from: ./saved_indices/lalonde_control_pop_control_voyager_20251025_192229
Loading model from: ./saved_indices/lalonde_control_pop_biasmod_20251025_192229.pkl


In [18]:
%%time
m_loaded = Matching("ATT", k=5, backend="voyager")
m_loaded.load_indices(
    control_index_path=control_path,
    bias_model_path=bias_path,
)

print("Index loaded! No re-indexing needed.")

Index loaded! No re-indexing needed.
CPU times: user 3.56 ms, sys: 4.98 ms, total: 8.54 ms
Wall time: 8.07 ms


In [19]:
%%time
est_loaded, se_loaded = m_loaded.estimate_with_preloaded(y, w, X)

CPU times: user 3 ms, sys: 4 ms, total: 7 ms
Wall time: 3.22 ms


In [20]:
print(f"\nATT Estimate (from loaded index): ${est_loaded:.2f} (SE: ${se_loaded:.2f})")

print("\nVerification:")
print(f"  Original: ${est:.2f}")
print(f"  Reloaded: ${est_loaded:.2f}")
print(f"  Match: {np.isclose(est, est_loaded)}")


ATT Estimate (from loaded index): $1696.51 (SE: $708.63)

Verification:
  Original: $1696.51
  Reloaded: $1696.51
  Match: True


### updating index with new observations

In [21]:
n_new_treated = 50
new_treated_indices = np.random.choice(
    np.where(w == 1)[0], n_new_treated, replace=False
)

y_new_treated = y[new_treated_indices]
X_new_treated = X[new_treated_indices]

control_indices = np.where(w == 0)[0]
full_data_w = np.concatenate([np.ones(n_new_treated), np.zeros(len(control_indices))])
full_data_y = np.concatenate([y_new_treated, y[control_indices]])
full_data_X = np.vstack([X_new_treated, X[control_indices]])

est_new, se_new = m_loaded.estimate_with_preloaded(
    full_data_y, full_data_w, full_data_X
)
print(f"\nATT for new cohort: ${est_new:.2f} (SE: ${se_new:.2f})")
print("No re-indexing of control population needed!")


ATT for new cohort: $1704.83 (SE: $1570.79)
No re-indexing of control population needed!


In [22]:
print("=" * 70)
print("Index files created:")
print("=" * 70)
for f in os.listdir("./saved_indices"):
    size = os.path.getsize(os.path.join("./saved_indices", f)) / 1024
    print(f"  {f}: {size:.1f} KB")


Index files created:
  lalonde_control_pop_control_voyager_20251025_192229.meta: 0.2 KB
  lalonde_control_pop_biasmod_20251025_192229.pkl: 0.5 KB
  lalonde_control_pop_control_voyager_20251025_192229.index: 2320.2 KB
